# Model 2 — Strategy Personalizer (SYNTHETIC)

Component 1 · Anchor AI (J26-IT-366). All results here are **SYNTHETIC**.

**Setup (run first).** Clones the C1 branch `dev/ravindu` of the team repo and installs Component 1's pinned requirements. If the repository is private, Colab asks you to sign in to GitHub (or use a personal access token in the URL). The cell also prints the Python version: C1 supports 3.11–3.13.

In [ ]:
import sys
print("Python", sys.version.split()[0])
!git clone --depth 1 -b dev/ravindu https://github.com/DulanjaR/AnchorAI.git
%cd AnchorAI
!pip install -q -r components/c1_disruption/requirements.txt

**Which data.** By default this uses the tiny example in `components/c1_disruption/sim/examples/`. To use your own persona task lists, upload `personas.csv` and `tasks.csv` (format: `components/c1_disruption/sim/README.md`) and change the two paths below. Everything produced here is **SYNTHETIC**: it checks that the pipeline works, it is not a research finding.

In [ ]:
PERSONAS = "components/c1_disruption/sim/examples/personas.csv"
TASKS = "components/c1_disruption/sim/examples/tasks.csv"
from components.c1_disruption.sim.personas import load_personas, load_tasks
personas = load_personas(PERSONAS)
tasks = load_tasks(TASKS, personas)
print(len(personas), "personas,", len(tasks), "planned tasks")

**Are the strategies really different?** The distinctness check plans every scenario with every strategy and reports how often each pair gives a different MVD. A pair that never differs would be merged or dropped.

In [ ]:
from components.c1_disruption.ml.bandit.actions import ACTIONS, distinctness
from components.c1_disruption.sim.generate import generate
from components.c1_disruption.sim.events import build_events
world = generate(personas, tasks, seed=11)
events, skipped = build_events(world, n_estimators=50)
print("strategies:", ACTIONS)
print(distinctness([e.engine_input for e in events]))
print(len(skipped), "events skipped: the engine cannot plan them (e.g. a None EF value, an open research decision)")

**Simulated acceptance.** Each SYNTHETIC student secretly prefers one strategy (`preferredStrategy`). We compare the fixed order, the per-student "most accepted" counter, random, and the pooled LinUCB bandit on the same events.

In [ ]:
from statistics import mean
from components.c1_disruption.sim.experiment import model2
curves = model2(world, events, seed=11)
for name, values in curves.items():
    print(f"{name:14} mean acceptance {mean(values) if values else float('nan'):.2f} over {len(values)} events")